# Lab | Combining and Structuring Data

**Dataset:** Customer insurance data (continuación del lab de Data Cleaning and Formatting)

Este laboratorio tiene dos partes:
1. **Combinar datos** — unir varios DataFrames en uno solo usando concat
2. **Estructurar datos** — usar pivot_table y melt para reorganizar la información y responder preguntas de negocio

## Importar librerías

In [2]:
import pandas as pd
import numpy as np

print('Librerías importadas correctamente')

Librerías importadas correctamente


## Challenge 1 — Combinar los datos

Cargamos los tres ficheros que forman parte del mismo conjunto de datos
(divididos en tres CSV distintos) y los combinamos en un único DataFrame.

In [3]:
# Cargamos los tres ficheros
url1 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv'
url2 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv'
url3 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv'

df1 = pd.read_csv(url1)
df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)

print(f'df1: {df1.shape}')
print(f'df2: {df2.shape}')
print(f'df3: {df3.shape}')

df1: (4008, 11)
df2: (996, 11)
df3: (7070, 11)


In [4]:
# Revisamos las columnas de cada DataFrame antes de combinarlos
print('Columnas df1:')
print(df1.columns.tolist())
print('\nColumnas df2:')
print(df2.columns.tolist())
print('\nColumnas df3:')
print(df3.columns.tolist())

Columnas df1:
['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Vehicle Class', 'Total Claim Amount']

Columnas df2:
['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Total Claim Amount', 'Policy Type', 'Vehicle Class']

Columnas df3:
['Customer', 'State', 'Customer Lifetime Value', 'Education', 'Gender', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Total Claim Amount', 'Vehicle Class']


### Estandarizar nombres de columnas

Antes de combinar, los nombres de las columnas deben coincidir entre los tres DataFrames.  
Aplicamos la misma limpieza de nombres que en el lab anterior: minúsculas, espacios por guiones bajos,
y renombramos `st` a `state` para que sea más descriptivo.

In [5]:
def clean_column_names(df):
    """
    Limpia los nombres de columnas: minúsculas y espacios por guiones bajos.
    """
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    return df

# Aplicamos la limpieza a los tres DataFrames
df1 = clean_column_names(df1)
df2 = clean_column_names(df2)
df3 = clean_column_names(df3)

# Renombramos 'st' a 'state' en los DataFrames que tengan esa columna
df1.rename(columns={'st': 'state'}, inplace=True)
df2.rename(columns={'st': 'state'}, inplace=True)

print('Columnas df1:', df1.columns.tolist())
print('Columnas df2:', df2.columns.tolist())
print('Columnas df3:', df3.columns.tolist())

Columnas df1: ['customer', 'state', 'gender', 'education', 'customer_lifetime_value', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'policy_type', 'vehicle_class', 'total_claim_amount']
Columnas df2: ['customer', 'state', 'gender', 'education', 'customer_lifetime_value', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'total_claim_amount', 'policy_type', 'vehicle_class']
Columnas df3: ['customer', 'state', 'customer_lifetime_value', 'education', 'gender', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'policy_type', 'total_claim_amount', 'vehicle_class']


In [6]:
# Comprobamos si las columnas ya coinciden entre los tres DataFrames
cols1 = set(df1.columns)
cols2 = set(df2.columns)
cols3 = set(df3.columns)

print('¿df1 y df2 tienen las mismas columnas?', cols1 == cols2)
print('¿df1 y df3 tienen las mismas columnas?', cols1 == cols3)

# Si hay diferencias, las mostramos
print('\nColumnas en df1 que no están en df3:', cols1 - cols3)
print('Columnas en df3 que no están en df1:', cols3 - cols1)

¿df1 y df2 tienen las mismas columnas? True
¿df1 y df3 tienen las mismas columnas? True

Columnas en df1 que no están en df3: set()
Columnas en df3 que no están en df1: set()


In [7]:
# Reordenamos las columnas de df2 y df3 para que coincidan con el orden de df1
# Esto no es obligatorio para concat, pero ayuda a revisar visualmente que todo está alineado
df2 = df2[df1.columns]
df3 = df3[df1.columns]

print('Columnas alineadas correctamente')

Columnas alineadas correctamente


### Combinar con concat

Usamos `pd.concat()` con `axis=0` para apilar los tres DataFrames verticalmente —
cada uno representa un grupo distinto de clientes con las mismas columnas.

In [8]:
# Concatenamos los tres DataFrames verticalmente (axis=0)
df = pd.concat([df1, df2, df3], axis=0, ignore_index=True)

print(f'Shape combinado: {df.shape}')
df.head()

Shape combinado: (12074, 11)


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


In [9]:
# Revisión general del DataFrame combinado
print('Tipos de datos:')
print(df.dtypes)
print('\nValores nulos por columna:')
print(df.isnull().sum())

Tipos de datos:
customer                      object
state                         object
gender                        object
education                     object
customer_lifetime_value       object
income                       float64
monthly_premium_auto         float64
number_of_open_complaints     object
policy_type                   object
vehicle_class                 object
total_claim_amount           float64
dtype: object

Valores nulos por columna:
customer                     2937
state                        2937
gender                       3059
education                    2937
customer_lifetime_value      2944
income                       2937
monthly_premium_auto         2937
number_of_open_complaints    2937
policy_type                  2937
vehicle_class                2937
total_claim_amount           2937
dtype: int64


In [10]:
# Eliminamos filas completamente vacías que puedan haber quedado de la concatenación
df.dropna(how='all', inplace=True)

# Eliminamos duplicados
n_antes = len(df)
df.drop_duplicates(inplace=True)
print(f'Filas duplicadas eliminadas: {n_antes - len(df)}')
print(f'Shape final: {df.shape}')

Filas duplicadas eliminadas: 3
Shape final: (9134, 11)


## Challenge 2 — Estructurar los datos

Ahora usamos `pivot_table` y `melt` para reorganizar los datos y responder
preguntas de negocio concretas.

### Pregunta 1 — Ingreso total por canal de ventas

Creamos una tabla pivote que muestre el ingreso total (`total_claim_amount`)
para cada canal de ventas (`sales_channel`), redondeado a 2 decimales.

In [13]:
# Pivot table: ingreso total por tipo de póliza (sales_channel no existe en este dataset)
pivot_poliza = df.pivot_table(
    index='policy_type',
    values='total_claim_amount',
    aggfunc='sum'
).round(2)

pivot_poliza.sort_values('total_claim_amount', ascending=False)

,total_claim_amount
policy_type,
Personal Auto,2926168.68
Corporate Auto,840081.32
Special Auto,165758.07


**Conclusión:** El canal con mayor ingreso total es el que aparece en la primera fila
tras ordenar — esto indica en qué canal la compañía recibe más reclamaciones en valor total,
lo que puede orientar decisiones sobre dónde reforzar recursos de atención al cliente.

In [15]:
# Ordenamos de mayor a menor para identificar el tipo de póliza principal
pivot_poliza.sort_values('total_claim_amount', ascending=False)

,total_claim_amount
policy_type,
Personal Auto,2926168.68
Corporate Auto,840081.32
Special Auto,165758.07


### Pregunta 2 — CLV medio por nivel educativo y género

Creamos una tabla pivote que muestre el Customer Lifetime Value (CLV) medio,
desglosado por nivel educativo (`education`) y género (`gender`).

In [18]:
# Limpiamos la columna customer_lifetime_value: quitamos '%' y convertimos a float
df['customer_lifetime_value'] = (
    df['customer_lifetime_value']
    .astype(str)
    .str.replace('%', '', regex=False)
    .astype(float)
)

print(df['customer_lifetime_value'].dtype)
print(df['customer_lifetime_value'].head())

float64
0           NaN
1     697953.59
2    1288743.17
3     764586.18
4     536307.65
Name: customer_lifetime_value, dtype: float64


In [19]:
# Pivot table: CLV medio por nivel educativo y género
pivot_clv = df.pivot_table(
    index='education',
    columns='gender',
    values='customer_lifetime_value',
    aggfunc='mean'
).round(2)

pivot_clv

gender,F,Femal,M,Male,female
education,,,,,
Bachelor,161710.09,628743.37,145606.03,1102121.81,916457.01
Bachelors,474208.57,NaN,520935.92,NaN,NaN
College,188319.11,666203.88,162121.79,796178.26,758341.72
Doctor,195166.78,NaN,158240.47,615315.91,NaN
High School or Below,176506.98,559589.48,162526.66,1115956.93,487828.18
Master,196807.59,NaN,150241.66,916723.23,475043.06


In [17]:
print(df['customer_lifetime_value'].dtype)
print(df['customer_lifetime_value'].head())

object
0            NaN
1     697953.59%
2    1288743.17%
3     764586.18%
4     536307.65%
Name: customer_lifetime_value, dtype: object


**Conclusión:** Comparando ambas columnas (Female / Male) por nivel educativo,
podemos ver si existen diferencias relevantes de CLV entre géneros dentro de cada
nivel formativo. Si las diferencias son pequeñas y no muestran un patrón claro,
esto sugiere que el nivel educativo y el género no son, por sí solos, buenos
predictores del valor de vida del cliente.

### Pregunta 3 — Importe total de reclamaciones por tipo de póliza y mes

Para analizar esta pregunta, primero pasamos los datos a formato largo con `melt`
y luego creamos una tabla pivote por tipo de póliza y mes.

In [21]:
# Comprobamos qué columnas relacionadas con fechas tenemos disponibles
print([col for col in df.columns if 'date' in col or 'month' in col])

['monthly_premium_auto']


In [23]:
# Pivot table: importe total de reclamaciones por tipo de póliza y clase de vehículo
pivot_poliza_vehiculo = df.pivot_table(
    index='policy_type',
    columns='vehicle_class',
    values='total_claim_amount',
    aggfunc='sum'
).round(2)

pivot_poliza_vehiculo

vehicle_class,Four-Door Car,Luxury Car,Luxury SUV,SUV,Sports Car,Two-Door Car
policy_type,,,,,,
Corporate Auto,346478.12,48933.97,35266.23,211677.10,57865.73,139860.19
Personal Auto,1204590.41,123032.88,156546.63,755952.44,194399.95,491646.38
Special Auto,67938.39,9625.45,5635.90,38610.72,17093.65,26853.96


In [25]:
# Pasamos la tabla pivote a formato largo con melt
pivot_reset = pivot_poliza_vehiculo.reset_index()

melted = pd.melt(
    pivot_reset,
    id_vars='policy_type',
    var_name='vehicle_class',
    value_name='total_claim_amount'
)

melted.head(10)

,policy_type,vehicle_class,total_claim_amount
0,Corporate Auto,Four-Door Car,346478.12
1,Personal Auto,Four-Door Car,1204590.41
2,Special Auto,Four-Door Car,67938.39
3,Corporate Auto,Luxury Car,48933.97
4,Personal Auto,Luxury Car,123032.88
5,Special Auto,Luxury Car,9625.45
6,Corporate Auto,Luxury SUV,35266.23
7,Personal Auto,Luxury SUV,156546.63
8,Special Auto,Luxury SUV,5635.90
9,Corporate Auto,SUV,211677.10


**Conclusión:** Al observar la tabla por tipo de póliza y mes, podemos identificar
si el tipo de póliza con mayor importe total de reclamaciones se mantiene constante
entre los distintos meses, o si varía. Si **Personal Auto** es consistentemente el tipo
con mayor importe en ambos meses, esto indica que es la categoría que genera más
reclamaciones de forma estable — relevante para la gestión de riesgo de la aseguradora.

In [27]:
# Identificamos la clase de vehículo con mayor importe total para cada tipo de póliza
for poliza in pivot_poliza_vehiculo.index:
    top_vehiculo = pivot_poliza_vehiculo.loc[poliza].idxmax()
    valor = pivot_poliza_vehiculo.loc[poliza].max()
    print(f'{poliza}: clase de vehículo con mayor importe total = {top_vehiculo} ({valor:.2f})')

Corporate Auto: clase de vehículo con mayor importe total = Four-Door Car (346478.12)
Personal Auto: clase de vehículo con mayor importe total = Four-Door Car (1204590.41)
Special Auto: clase de vehículo con mayor importe total = Four-Door Car (67938.39)
